# Stage 5: testing the models on GOTOV (older adults, a different sensor)

Every model so far was trained **and** tested on Capture-24. GOTOV is a completely separate dataset:

| | Capture-24 | GOTOV |
|---|---|---|
| People | 151, ages 18 to 53+ | **35, ages 60 to 81** |
| Sensor | Axivity AX3, dominant wrist, 100 Hz | **GENEActiv, right wrist, about 86 Hz** |
| Setting | a normal day, labelled from a body camera | **a set routine in and around a lab**, timed by a researcher |

So this asks: **do the models still work on older people, wearing a different sensor, who were never part of training?**

**Nothing is trained or tuned here.** The Stage 1 to 3 models and the Stage 1 HMM are used exactly as saved.

1. Read the GOTOV wrist files and match each activity to our 4 classes
2. Process them the same way as Capture-24: 50 Hz, 30 s windows
3. Check how GOTOV's signals compare with Capture-24's
4. Predict with all 6 models, and score them
5. Which activities go wrong?

**Kernel:** `.venv-tf` in this folder. The forest runs through `rf_predict.py` with anaconda's Python (see section 4).

## How this works (for beginners)

**Why test on another dataset?** Cross-validation tested on new *people*, but all from the same study: same sensor, same kind of day, same way of labelling. A real watch is a new sensor on new people in a new setting. Testing on a dataset the models have never seen is the closest we can get to that before the real watch. This is called **external validation**.

**What can go wrong when the data changes** (*domain shift*):

| Change | Why it can hurt |
|---|---|
| Older people | they may move more slowly or less, so walking can look like light activity |
| A different sensor | each brand points its x, y, z axes differently, and is calibrated slightly differently. Features that use the wrist *angle* (pitch, roll, the mean of each axis) can be misread |
| A scripted routine | activities change every few minutes, much faster than in a normal day, which the HMM did not learn |
| No sleep | GOTOV has nobody asleep, only lying awake, which our models may call sleep |

**How we score it.** GOTOV has no sleep, so three views:

1. **Still vs moving**: the main question for the watch
2. **3 classes** (sedentary, light, moderate-vigorous), with any "sleep" answer counted as sedentary. Lying awake and asleep look almost the same on a wrist, so this is fair
3. **4 classes, strict**: "sleep" for someone lying awake counts as wrong

In [1]:
import subprocess
import time
import zipfile
from pathlib import Path

import keras
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.signal import butter, sosfiltfilt
from sklearn.metrics import f1_score, recall_score

from common import (BINARY, CLASSES, RESULTS_DIR, confusion_table, make_features, paired_bootstrap, scores, smooth,
                    to_binary)

GOTOV_DIR = Path('../../data/open datasets/GOTOV Human Physical Activity and Energy Expenditure Dataset on Older Individuals_2_all')
OUT = Path('processed/gotov')
OUT.mkdir(exist_ok=True)
HZ, WIN = 50, 1500
ANACONDA = Path.home() / 'anaconda3' / 'python.exe'
print('Keras', keras.__version__, 'backend', keras.config.backend(), '| XGBoost', xgb.__version__)

Keras 3.15.1 backend tensorflow | XGBoost 3.2.0


## 1. GOTOV's activities, matched to our 4 classes

GOTOV labels 16 activities. Each is matched to the class **Capture-24 itself** gives a similar activity (`annotation-label-dictionary.csv`, column `label:Walmsley2020`), so the rules are the same as in training:

| GOTOV activity | Our class | Similar Capture-24 activity and its class |
|---|---|---|
| lying down (left, right), sitting (sofa, couch, chair) | sedentary | "sitting/lying reading or watching TV" → sedentary |
| standing | light | "standing in a line, MET 1.3" → light (Capture-24 counts standing as light) |
| dishwashing, stacking shelves, vacuuming | light | "kitchen activity", "household chores MET 2.5", "sweeping carpet or floors" → light |
| walking slow | light | "walking household, MET 2.0" → light |
| walking normal, walking fast | moderate-vigorous | "walking as the single means to a destination, MET 3.0" → moderate-vigorous |
| cycling, stairs up, step, jumping | moderate-vigorous | "bicycling MET 4.0", "walking upstairs MET 4.0" → moderate-vigorous |

**Judgement calls:** *walking slow* (an older person's slow walk is around 2–3 MET) and *stacking shelves* (Capture-24 has "putting away items" as light but "packing/unpacking with lifting" as moderate-vigorous) could reasonably go either way. Both are moving in still vs moving, so the main result doesn't depend on them.

In [2]:
MAPPING = {'lyingDownLeft': 'sedentary', 'lyingDownRight': 'sedentary', 'sittingSofa': 'sedentary',
           'sittingCouch': 'sedentary', 'sittingChair': 'sedentary',
           'standing': 'light', 'dishwashing': 'light', 'stakingShelves': 'light', 'vacuumCleaning': 'light',
           'walkingSlow': 'light',
           'walkingNormal': 'moderate-vigorous', 'walkingFast': 'moderate-vigorous', 'cycling': 'moderate-vigorous',
           'walkingStairsUp': 'moderate-vigorous', 'step': 'moderate-vigorous', 'syncJumping': 'moderate-vigorous'}
ACTIVITIES = list(MAPPING)                         # (stakingShelves is GOTOV's own spelling)
assert set(MAPPING.values()) <= set(CLASSES)

archive = zipfile.ZipFile(GOTOV_DIR / 'Activity_Measurements.zip')
members = sorted(n for n in archive.namelist() if n.endswith('-GENEActiv-wrist.csv') and not n.startswith('__MACOSX'))
people = [n.split('/')[1] for n in members]
info = pd.read_csv(GOTOV_DIR / 'GOTOV_DataSummary.csv').set_index('ID')
print(f'{len(people)} wrist files. Age {info.age.min()} to {info.age.max()}, median {info.age.median()}')

35 wrist files. Age 59.5 to 81.3, median 64.6


## 2. The same processing as Capture-24

Capture-24 (Stage 0): low-pass filter at 20 Hz, then every 2nd sample (100 → 50 Hz), then 30 s windows starting at the beginning of each labelled stretch.

GOTOV is about 86 Hz, so "every 2nd sample" doesn't work. Instead, for each labelled stretch (one activity, done once):

1. **Rebuild even timestamps.** GOTOV stores time in whole milliseconds, so a sample every 11.67 ms shows up as steps of 11 or 12 ms. The true rate is the number of samples divided by the stretch's length. A check makes sure the stored times never stray more than 20 ms from the rebuilt ones (no hidden gaps)
2. **Low-pass filter at 20 Hz**, the same filter as Stage 0 (WHY: removes detail faster than 25 Hz, which 50 Hz can't hold)
3. **Resample to exactly 50 Hz** by straight-line interpolation between neighbouring samples
4. **Cut 30 s windows** from the start of the stretch. The leftover piece at the end is dropped, as in Capture-24

Every window is one activity from start to end. Stretches under 30 s (stairs, jumping, most "step" stretches) give no windows.

In [3]:
def bouts(labels, t):
    """(start, end) row ranges of each labelled stretch: same label, no gap over 100 ms."""
    change = np.flatnonzero((labels[1:] != labels[:-1]) | (np.diff(t) > 100)) + 1
    edges = np.r_[0, change, len(labels)]
    return [(a, b) for a, b in zip(edges[:-1], edges[1:]) if labels[a] != 'NA']

def to_50hz(t, xyz):
    """One stretch at about 86 Hz -> (start time in ms, samples at exactly 50 Hz)."""
    n = len(t)
    fs = (n - 1) / ((t[-1] - t[0]) / 1000)
    even = t[0] + np.arange(n) * 1000 / fs                      # the timestamps the sensor really meant
    assert np.abs(t - even).max() < 20, 'gap or jump in the timestamps'
    smooth_xyz = sosfiltfilt(butter(4, 20, fs=fs, output='sos'), xyz, axis=0)
    grid = t[0] + np.arange(int((n - 1) / fs * HZ) + 1) * 1000 / HZ
    return t[0], np.column_stack([np.interp(grid, even, smooth_xyz[:, k]) for k in range(3)]), fs

def process_person(member):
    pid = member.split('/')[1]
    d = pd.read_csv(archive.open(member), usecols=['time', 'x', 'y', 'z', 'labels'], low_memory=False)
    t = d['time'].to_numpy('float64')
    labels = d['labels'].fillna('NA').astype(object).to_numpy()
    assert np.all(np.diff(t) > 0) and set(labels) - {'NA'} <= set(MAPPING), set(labels) - set(MAPPING)
    X, activity, start, rates = [], [], [], []
    for a, b in bouts(labels, t):
        if t[b - 1] - t[a] < WIN * 1000 / HZ:          # under 30 s: no full window, skip it
            continue
        t0, xyz, fs = to_50hz(t[a:b], d[['x', 'y', 'z']].to_numpy('float64')[a:b])
        rates.append(fs)
        for w in range(len(xyz) // WIN):
            X.append(xyz[w * WIN:(w + 1) * WIN]); activity.append(labels[a]); start.append(t0 + w * WIN * 1000 / HZ)
    X = np.array(X, dtype='float32').reshape(-1, WIN, 3)
    y4 = np.array([CLASSES.index(MAPPING[a]) for a in activity], dtype='int8')
    np.savez(OUT / f'{pid}.npz', X=X, y4=y4, activity=np.array(activity, dtype=str), start=np.array(start, dtype='int64'))
    return {'pid': pid, 'windows': len(X), 'median Hz': np.median(rates) if rates else np.nan}

# check on fake data: a 2 Hz wave at 86 Hz comes out as a 2 Hz wave at 50 Hz, with the same size
fake_t = 1e12 + np.round(np.arange(86 * 60) * 1000 / 86)       # whole milliseconds, like GOTOV
wave = np.sin(2 * np.pi * 2 * (fake_t - fake_t[0]) / 1000)
_, out, fs = to_50hz(fake_t, np.column_stack([wave, wave, wave]))
expected = np.sin(2 * np.pi * 2 * np.arange(len(out)) / HZ)
assert abs(fs - 86) < 0.01 and np.abs(out[100:-100, 0] - expected[100:-100]).max() < 0.01
print('resampling check passed')

t0 = time.time()
summary = pd.DataFrame([process_person(m) for m in members]) if not all((OUT / f'{p}.npz').exists() for p in people) else None
if summary is not None:
    summary.to_csv(OUT / 'summary.csv', index=False)
summary = pd.read_csv(OUT / 'summary.csv')
print(f'{summary.windows.sum():,} windows from {len(summary)} people ({time.time() - t0:.0f} s). '
      f'Sampling rate: median {summary["median Hz"].median():.2f} Hz')

resampling check passed
2,822 windows from 35 people (0 s). Sampling rate: median 85.71 Hz


In [4]:
# load everything (small: a few thousand windows)
X, y, act, pid, start = [], [], [], [], []
for p in people:
    with np.load(OUT / f'{p}.npz') as f:
        order = np.argsort(f['start'])                 # time order within each person, for the HMM
        X.append(f['X'][order]); y.append(f['y4'][order]); act.append(f['activity'][order]); start.append(f['start'][order])
    pid.append(np.full(len(y[-1]), p))
X, y, act, pid, start = (np.concatenate(v) for v in (X, y, act, pid, start))
y = y.astype(int)
assert X.shape == (len(y), WIN, 3) and np.isfinite(X).all()

counts = (pd.DataFrame({'activity': act, 'pid': pid}).groupby('activity')
          .agg(windows=('pid', 'size'), people=('pid', 'nunique')).reindex(ACTIVITIES).fillna(0).astype(int))
counts['our class'] = [MAPPING[a] for a in counts.index]
print(counts.to_string())
print('\nwindows per class:', dict(zip(CLASSES, np.bincount(y, minlength=4))), '| still vs moving:', np.bincount(to_binary(y)))

                 windows  people          our class
activity                                           
lyingDownLeft        209      35          sedentary
lyingDownRight       206      35          sedentary
sittingSofa          206      35          sedentary
sittingCouch         210      35          sedentary
sittingChair         201      34          sedentary
standing             130      35              light
dishwashing          209      35              light
stakingShelves       192      32              light
vacuumCleaning       190      32              light
walkingSlow          208      21              light
walkingNormal        207      21  moderate-vigorous
walkingFast          195      21  moderate-vigorous
cycling              430      18  moderate-vigorous
walkingStairsUp        0       0  moderate-vigorous
step                  29      29  moderate-vigorous
syncJumping            0       0  moderate-vigorous

windows per class: {'sleep': np.int64(0), 'sedentary': np.int64

## 3. How does GOTOV's signal compare with Capture-24's?

Same features as Stages 1 and 3, compared class by class. If GOTOV's values sit far from Capture-24's for the same class, the models will be reading numbers they never saw in training.

- **Movement features** (`mag_std`, power) depend on how people move, but not on how the sensor is mounted
- **Angle features** (`x_mean`, `y_mean`, `z_mean`, `pitch`, `roll`) depend on which way the sensor's axes point, which differs between brands

In [5]:
F = make_features(X)
np.savez(OUT / 'features.npz', F=F)
from common import FEATURES
with np.load('processed/features_all.npz') as f:
    F_c24, y_c24 = f['F'], f['y']

cols = ['mag_mean', 'mag_std', 'power_0.5_3', 'x_mean', 'y_mean', 'z_mean', 'pitch', 'roll']
rows = []
for k in [1, 2, 3]:
    for name, FF, yy in [('Capture-24', F_c24, y_c24), ('GOTOV', F, y)]:
        rows.append({'class': CLASSES[k], 'dataset': name, **{col: np.median(FF[yy == k, FEATURES.index(col)]) for col in cols}})
print('median of each feature:')
print(pd.DataFrame(rows).set_index(['class', 'dataset']).round(3).to_string())

median of each feature:
                              mag_mean  mag_std  power_0.5_3  x_mean  y_mean  z_mean   pitch     roll
class             dataset                                                                            
sedentary         Capture-24     1.002    0.046        0.001  -0.465   0.040   0.317  30.243    4.917
                  GOTOV          1.016    0.008        0.000  -0.521   0.019  -0.444  30.886    4.513
light             Capture-24     1.023    0.164        0.013  -0.521  -0.125   0.077  39.338  -34.810
                  GOTOV          1.051    0.224        0.025  -0.562   0.647  -0.226  37.090  103.018
moderate-vigorous Capture-24     1.044    0.275        0.036  -0.220  -0.325   0.114  12.829  -40.867
                  GOTOV          1.071    0.278        0.027  -0.482   0.641  -0.364  27.791  110.828


## 4. Predict with all six models

- **RF:** the Stage 1 forest. It was saved by an older scikit-learn (anaconda), which the newer one here can't open, so `rf_predict.py` runs it in anaconda. That script first checks it reproduces the Stage 1 predictions exactly
- **CNN:** the Stage 2 model (`cnn_4class.keras`)
- **XGB:** the Stage 3 model (`xgb_4class.json`)
- **HMM:** the Stage 1 HMM (`hmm.npz`), learned from Capture-24, applied to each GOTOV person's windows in time order

Prediction only, nothing is trained. It takes seconds.

In [6]:
print(subprocess.run([str(ANACONDA), 'rf_predict.py', str(OUT / 'features.npz'), str(OUT / 'rf_probs.npy')], check=True,
                     capture_output=True, text=True).stdout)
probs = {'RF': np.load(OUT / 'rf_probs.npy')}
probs['CNN'] = keras.models.load_model(RESULTS_DIR / 'cnn_4class.keras').predict(X, batch_size=256, verbose=0)
booster = xgb.XGBClassifier()
booster.load_model(RESULTS_DIR / 'xgb_4class.json')
probs['XGB'] = booster.predict_proba(F)

hmm = np.load(RESULTS_DIR / 'hmm.npz')
pred = {}
for name in ['RF', 'CNN', 'XGB']:
    assert probs[name].shape == (len(y), 4) and np.allclose(probs[name].sum(axis=1), 1, atol=1e-4)
    pred[name] = probs[name].argmax(axis=1)
    pred[name + '+HMM'] = smooth(probs[name], pid, hmm['start'], hmm['transition'])
MODELS = ['RF', 'RF+HMM', 'CNN', 'CNN+HMM', 'XGB', 'XGB+HMM']
pred = {m: pred[m] for m in MODELS}
np.savez(RESULTS_DIR / 'stage5_gotov_predictions.npz', pid=pid, start=start, y=y, activity=act,
         **{f'probs_{n}': p for n, p in probs.items()}, **{f'pred_{n}': p for n, p in pred.items()})
print('predicted', len(y), 'windows with', len(pred), 'models')

forest reproduces Stage 1; predicted 2,822 windows



predicted 2822 windows with 6 models


## 5. Scores

`3-class` merges "sleep" into sedentary (GOTOV has nobody asleep). `4-class strict` counts "sleep" as wrong. For comparison, the Capture-24 cross-validation scores are on the right (those have sleep, and many more still windows, so they're a rough guide, not an exact match).

In [7]:
to3 = lambda v: np.maximum(np.asarray(v), 1) - 1     # sleep -> sedentary, then 0 = sedentary, 1 = light, 2 = MVPA
rows = []
for m in MODELS:
    rows.append({'model': m, 'task': 'binary', **scores(to_binary(y), to_binary(pred[m]), 2)})
    rows.append({'model': m, 'task': '3-class', **scores(to3(y), to3(pred[m]), 3)})
    strict = scores(y, pred[m], 4)
    rows.append({'model': m, 'task': '4-class strict', 'accuracy': strict['accuracy'], 'kappa': strict['kappa']})
results = pd.DataFrame(rows)
results.to_csv(RESULTS_DIR / 'stage5_gotov_metrics.csv', index=False)
table = results.pivot(index='model', columns='task', values=['accuracy', 'macro-F1', 'kappa']).loc[MODELS]
table = table.drop(columns=[('macro-F1', '4-class strict')])
cv = pd.read_csv(RESULTS_DIR / 'cv_metrics.csv').pivot(index='model', columns='task', values='kappa').loc[MODELS]
table[('Capture-24 CV kappa', 'binary')] = cv['binary']
table[('Capture-24 CV kappa', '4-class')] = cv['4-class']
print(table.round(3).to_string())

        accuracy                       macro-F1          kappa                       Capture-24 CV kappa        
task     3-class 4-class strict binary  3-class binary 3-class 4-class strict binary              binary 4-class
model                                                                                                           
RF         0.769          0.699  0.920    0.757  0.916   0.650          0.562  0.832               0.723   0.707
RF+HMM     0.833          0.833  0.945    0.826  0.941   0.748          0.748  0.883               0.799   0.825
CNN        0.771          0.590  0.956    0.762  0.952   0.654          0.440  0.905               0.751   0.683
CNN+HMM    0.799          0.641  0.960    0.792  0.957   0.698          0.505  0.914               0.802   0.788
XGB        0.757          0.692  0.922    0.752  0.915   0.634          0.553  0.830               0.730   0.707
XGB+HMM    0.820          0.820  0.952    0.816  0.947   0.729          0.729  0.895            

In [8]:
print('how often each model calls lying-awake people "asleep":')
lying = np.isin(act, ['lyingDownLeft', 'lyingDownRight'])
print({m: f'{100 * (pred[m][lying] == 0).mean():.0f}%' for m in MODELS})
for m in ['RF+HMM', 'CNN+HMM', 'XGB+HMM']:
    print(f'\n===== {m}: still vs moving =====')
    print(confusion_table(to_binary(y), to_binary(pred[m]), BINARY).to_string())

how often each model calls lying-awake people "asleep":
{'RF': '39%', 'RF+HMM': '0%', 'CNN': '70%', 'CNN+HMM': '67%', 'XGB': '32%', 'XGB+HMM': '0%'}

===== RF+HMM: still vs moving =====
             still  moving  still %  moving %
true still     999      33     96.8       3.2
true moving    123    1667      6.9      93.1

===== CNN+HMM: still vs moving =====
             still  moving  still %  moving %
true still     961      71     93.1       6.9
true moving     41    1749      2.3      97.7

===== XGB+HMM: still vs moving =====
             still  moving  still %  moving %
true still     936      96     90.7       9.3
true moving     40    1750      2.2      97.8


## 6. Which activities go wrong?

For each GOTOV activity: the share of its windows each model gets right. "Right" = the class it is matched to in section 1, with "sleep" counted as sedentary. The still-vs-moving column says whether at least the still/moving answer was right.

In [9]:
rows = []
for a in [a for a in ACTIVITIES if (act == a).any()]:
    m_ = act == a
    row = {'activity': a, 'windows': int(m_.sum()), 'our class': MAPPING[a]}
    for m in ['RF+HMM', 'CNN+HMM', 'XGB+HMM']:
        row[f'{m} 3-class'] = 100 * (to3(pred[m][m_]) == to3(y[m_])).mean()
        row[f'{m} still/moving'] = 100 * (to_binary(pred[m][m_]) == to_binary(y[m_])).mean()
    rows.append(row)
by_activity = pd.DataFrame(rows).set_index('activity')
by_activity.to_csv(RESULTS_DIR / 'stage5_gotov_by_activity.csv')
print('% of windows right:')
print(by_activity.round(0).to_string())

print('\nwhat XGB+HMM says for each activity (% of its windows):')
answer = pd.crosstab(pd.Series(act, name='activity'), pd.Series(np.array(CLASSES)[pred['XGB+HMM']], name='XGB+HMM says'),
                     normalize='index').reindex(index=by_activity.index, columns=CLASSES, fill_value=0) * 100
print(answer.round(0).to_string())

% of windows right:
                windows          our class  RF+HMM 3-class  RF+HMM still/moving  CNN+HMM 3-class  CNN+HMM still/moving  XGB+HMM 3-class  XGB+HMM still/moving
activity                                                                                                                                                     
lyingDownLeft       209          sedentary           100.0                100.0             99.0                  99.0             97.0                  97.0
lyingDownRight      206          sedentary           100.0                100.0            100.0                 100.0            100.0                 100.0
sittingSofa         206          sedentary           100.0                100.0            100.0                 100.0            100.0                 100.0
sittingCouch        210          sedentary           100.0                100.0             92.0                  92.0             99.0                  99.0
sittingChair        201         

## 7. Person by person

35 people, each scored on their own. Which model is better, and is the gap more than luck?

In [10]:
rows = []
for p in people:
    m_ = pid == p
    if not m_.any():
        continue
    row = {'pid': p, 'age': info.age[p]}
    for m in MODELS:
        row[f'{m} binary'] = scores(to_binary(y[m_]), to_binary(pred[m][m_]), 2)['kappa']
        row[f'{m} 3-class'] = scores(to3(y[m_]), to3(pred[m][m_]), 3)['kappa']
    rows.append(row)
per_person = pd.DataFrame(rows)
per_person.to_csv(RESULTS_DIR / 'stage5_gotov_per_person.csv', index=False)
print(f'{len(per_person)} people. Median kappa per person:')
print(pd.DataFrame({t: {m: per_person[f'{m} {t}'].median() for m in MODELS} for t in ['binary', '3-class']}).round(3).to_string())

rows = []
for a, b in [('XGB+HMM', 'RF+HMM'), ('RF+HMM', 'CNN+HMM'), ('XGB+HMM', 'CNN+HMM'), ('RF+HMM', 'RF'), ('CNN+HMM', 'CNN'), ('XGB+HMM', 'XGB')]:
    for t in ['binary', '3-class']:
        both = per_person[[f'{a} {t}', f'{b} {t}']].dropna()
        mean, lo, hi = paired_bootstrap(both.iloc[:, 0], both.iloc[:, 1])
        rows.append({'A': a, 'B': b, 'task': t, 'A better for': f'{int((both.iloc[:, 0] > both.iloc[:, 1]).sum())} / {len(both)}',
                     'mean kappa A - B': round(mean, 3), '95% range': f'{lo:+.3f} to {hi:+.3f}',
                     'verdict': 'A better' if lo > 0 else ('B better' if hi < 0 else 'tie')})
pairwise = pd.DataFrame(rows)
pairwise.to_csv(RESULTS_DIR / 'stage5_gotov_pairwise.csv', index=False)
print()
print(pairwise.to_string(index=False))

35 people. Median kappa per person:
         binary  3-class
RF        0.842    0.634
RF+HMM    0.888    0.749
CNN       0.911    0.641
CNN+HMM   0.957    0.694
XGB       0.813    0.647
XGB+HMM   0.874    0.753

      A       B    task A better for  mean kappa A - B        95% range  verdict
XGB+HMM  RF+HMM  binary      15 / 35             0.038 -0.011 to +0.107      tie
XGB+HMM  RF+HMM 3-class      14 / 35            -0.004 -0.062 to +0.069      tie
 RF+HMM CNN+HMM  binary       8 / 35            -0.051 -0.131 to +0.017      tie
 RF+HMM CNN+HMM 3-class      20 / 35             0.033 -0.050 to +0.101      tie
XGB+HMM CNN+HMM  binary       9 / 35            -0.013 -0.050 to +0.030      tie
XGB+HMM CNN+HMM 3-class      20 / 35             0.029 -0.021 to +0.079      tie
 RF+HMM      RF  binary      22 / 35             0.032 -0.024 to +0.081      tie
 RF+HMM      RF 3-class      30 / 35             0.087 +0.034 to +0.131 A better
CNN+HMM     CNN  binary      20 / 35             0.014 -0.0

## 8. What this shows

**1. Still vs moving carries over well.** With the HMM, binary kappa is 0.88–0.91 on GOTOV, even higher than on Capture-24 (0.80–0.81). But GOTOV is easier: its activities are scripted and clean, and its still windows are much quieter (median `mag_std` 0.008 g for sedentary, against 0.046 g in Capture-24's daily life). So this shows the models work on older people with a different sensor, **not** that they'd score this high in older people's daily lives.

**2. The finer classes are harder.** 3-class kappa is 0.70–0.75 with the HMM. The main error: **older people's normal and fast walking is often called light** (XGB+HMM: 60% and 50% of those windows). Their walking may simply be gentler, or the "moderate-vigorous" line (about 3 MET) sits differently for them. Cycling is almost always right (94–99%).

**3. Each model has its own weak spot:**
- **CNN:** calls people lying awake "asleep" 67% of the time, even with the HMM (the tree models + HMM: 0%). Fine for still vs moving, wrong for 4 classes
- **XGBoost:** calls sitting at a computer "light" 43% of the time (typing), which gives it the most false alarms (9% of still windows)
- **Forest:** calls standing "sedentary" 76% of the time

**4. With 35 people, no model is clearly better than another.** Every model-vs-model comparison is a tie (all 95% ranges include 0). The CNN + HMM has the best still-vs-moving score here (0.914), the opposite of Capture-24, but it is not a significant difference.

**5. The HMM helps again,** clearly on 3 classes for all three models.

**6. The sensor's axes point differently** (section 3: `z_mean` flips sign when sitting, `roll` is about 140° off), yet the models still work, because movement features carry most of the decision. The SilverGuard MPU-6050 will have its own axis directions, so the watch will need its axes lined up with the training data (and calibrated) before export.

In [11]:
# every claim above, checked against the numbers
r = results.set_index(['model', 'task'])['kappa']
hmm_models = ['RF+HMM', 'CNN+HMM', 'XGB+HMM']
assert all(0.88 <= r[m, 'binary'] <= 0.915 for m in hmm_models)                               # 1
assert all(r[m, 'binary'] > cv.loc[m, 'binary'] for m in hmm_models)
c24_sed = np.median(F_c24[y_c24 == 1, FEATURES.index('mag_std')]); gotov_sed = np.median(F[y == 1, FEATURES.index('mag_std')])
assert round(gotov_sed, 3) == 0.008 and round(c24_sed, 3) == 0.046
assert all(0.69 <= r[m, '3-class'] <= 0.75 for m in hmm_models)                              # 2
assert round(answer.loc['walkingNormal', 'light']) == 60 and round(answer.loc['walkingFast', 'light']) == 50
assert by_activity.loc['cycling', [f'{m} 3-class' for m in hmm_models]].min() >= 94
assert round(100 * (pred['CNN+HMM'][lying] == 0).mean()) == 67                                 # 3
assert (pred['RF+HMM'][lying] == 0).sum() == 0 and (pred['XGB+HMM'][lying] == 0).sum() == 0
assert round(answer.loc['sittingChair', 'light']) == 43
false_alarm = {m: ((to_binary(pred[m]) == 1) & (y <= 1)).sum() / (y <= 1).sum() for m in hmm_models}
assert max(false_alarm, key=false_alarm.get) == 'XGB+HMM' and round(100 * false_alarm['XGB+HMM']) == 9
assert round(100 - by_activity.loc['standing', 'RF+HMM 3-class']) == 76
v = pairwise.set_index(['A', 'B', 'task'])['verdict']                                          # 4
assert all(v[a, b, t] == 'tie' for a, b in [('XGB+HMM', 'RF+HMM'), ('RF+HMM', 'CNN+HMM'), ('XGB+HMM', 'CNN+HMM')] for t in ['binary', '3-class'])
assert max(hmm_models, key=lambda m: r[m, 'binary']) == 'CNN+HMM'
assert all(v[m, m.replace('+HMM', ''), '3-class'] == 'A better' for m in hmm_models)          # 5
print('every claim above matches the numbers')

every claim above matches the numbers


## Update after Stage 6

**Finding 2 above was revised.** Stage 6 measured GOTOV's walking cadence from the chest sensor: median 107 (slow), 116 (normal) and 125 (fast) steps/min, so the walking was not slow. The main reason it was called *light* was the **sensor's axes**: GOTOV's wrist sensor is turned half a turn around its x axis compared with Capture-24's. With the axes lined up (chosen without labels), still vs moving kappa rises to 0.97 (forest) and 0.99 (XGBoost), and normal walking is called moderate-vigorous 91% of the time. The desk-sitting and standing errors in finding 3 mostly disappear too. See `stage6_har70_walking.ipynb`, sections 5 to 7.